# AI Threat Detection — Exploratory Data Analysis

This notebook provides:
- Dataset exploration and statistics
- Class distribution visualization
- Feature correlation analysis
- Model performance visualization

**Dataset:** UNSW-NB15 Network Intrusion Detection Dataset  
**Model:** LSTM Binary Classifier (Normal vs Attack)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Add project root to path so we can import utils
sys.path.insert(0, os.path.abspath('..'))

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

print('All libraries loaded successfully.')

---
## Section 1 — Dataset Exploration

In [ ]:
# Load the raw training CSV
TRAIN_CSV = '../dataset/UNSW_NB15_training-set.csv'
TEST_CSV  = '../dataset/UNSW_NB15_testing-set.csv'

try:
    train_df = pd.read_csv(TRAIN_CSV)
    test_df  = pd.read_csv(TEST_CSV)
    print(f'Training set : {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
    print(f'Testing set  : {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
except FileNotFoundError:
    print('CSV files not found. Place UNSW-NB15 CSVs in the dataset/ folder.')
    print('Using a small synthetic demo dataset for notebook illustration.')
    # Synthetic demo data for notebook illustration without the real dataset
    np.random.seed(42)
    n = 1000
    feature_cols = [f'feat_{i}' for i in range(40)]
    train_df = pd.DataFrame(np.random.randn(n, 40), columns=feature_cols)
    train_df['label'] = np.random.randint(0, 2, n)
    train_df['proto'] = np.random.choice(['tcp','udp','icmp'], n)
    train_df['service'] = np.random.choice(['http','ftp','smtp','-'], n)
    train_df['state'] = np.random.choice(['FIN','INT','CON','REQ'], n)
    test_df = train_df.sample(200).reset_index(drop=True)
    print(f'Demo dataset: {train_df.shape}')

In [ ]:
# Preview the first few rows
print('First 5 rows of training data:')
train_df.head()

In [ ]:
# Data types and non-null counts
print('Dataset Info:')
train_df.info()

In [ ]:
# Statistical summary of numeric columns
print('Statistical Summary (numeric features):')
train_df.describe().round(3)

In [ ]:
# Check for missing values
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values found in the training set.')
else:
    print(f'Columns with missing values:')
    display(missing_df)

---
## Section 2 — Class Distribution

In [ ]:
# Class counts
class_counts = train_df['label'].value_counts().sort_index()
class_labels = ['Normal (0)', 'Attack (1)']
colors = ['#2196F3', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Bar chart ──────────────────────────────────────────────
bars = axes[0].bar(class_labels, class_counts.values, color=colors, width=0.5, edgecolor='white')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Distribution — Count', fontsize=13)
axes[0].set_ylabel('Number of Samples')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Pie chart ──────────────────────────────────────────────
axes[1].pie(class_counts.values, labels=class_labels, colors=colors,
            autopct='%1.1f%%', startangle=140, pctdistance=0.75,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution — Proportion', fontsize=13)

plt.suptitle('UNSW-NB15 Class Balance', fontsize=15, y=1.01)
plt.tight_layout()

os.makedirs('../results/graphs', exist_ok=True)
plt.savefig('../results/graphs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nClass Counts:\n{class_counts.to_string()}')

---
## Section 3 — Feature Analysis

In [ ]:
# Categorical feature distributions
cat_cols = train_df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')

if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        vc = train_df[col].value_counts().head(10)
        vc.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(f'Top Values: {col}', fontsize=12)
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=45)
    plt.suptitle('Categorical Feature Distributions', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('../results/graphs/categorical_features.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Distribution of selected numeric features split by class
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'label']

# Pick top 6 most variable features
top_features = train_df[numeric_cols].std().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, top_features):
    normal = train_df[train_df['label'] == 0][col].clip(upper=train_df[col].quantile(0.99))
    attack = train_df[train_df['label'] == 1][col].clip(upper=train_df[col].quantile(0.99))
    ax.hist(normal, bins=40, alpha=0.6, color='#2196F3', label='Normal', density=True)
    ax.hist(attack, bins=40, alpha=0.6, color='#F44336', label='Attack', density=True)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Feature Distributions: Normal vs Attack', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/graphs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap — top 15 most correlated features with label
numeric_df = train_df[numeric_cols + ['label']].copy()
corr_with_label = numeric_df.corr()['label'].drop('label').abs().nlargest(15)
top15_cols = corr_with_label.index.tolist() + ['label']
corr_matrix = numeric_df[top15_cols].corr()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap (Top 15 + Label)', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../results/graphs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Model Performance Visualization

In [ ]:
# Load saved training history graph if it exists
history_img = '../results/graphs/training_history.png'
if os.path.exists(history_img):
    from IPython.display import Image, display as ipy_display
    print('Training History:')
    ipy_display(Image(history_img))
else:
    print('Training history not yet available.')
    print('Run  python model/train.py  to train the model first.')

In [ ]:
# Load saved confusion matrix if it exists
cm_img = '../results/graphs/confusion_matrix.png'
if os.path.exists(cm_img):
    from IPython.display import Image, display as ipy_display
    print('Confusion Matrix:')
    ipy_display(Image(cm_img))
else:
    print('Confusion matrix not yet available.')
    print('Run  python model/train.py  to train the model first.')

In [ ]:
# Analyze processed data shapes if available
processed_dir = '../dataset/processed'

files = ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy']
if all(os.path.exists(os.path.join(processed_dir, f)) for f in files):
    X_train = np.load(os.path.join(processed_dir, 'X_train.npy'))
    X_test  = np.load(os.path.join(processed_dir, 'X_test.npy'))
    y_train = np.load(os.path.join(processed_dir, 'y_train.npy'))
    y_test  = np.load(os.path.join(processed_dir, 'y_test.npy'))

    print('Processed Dataset Summary')
    print('=' * 40)
    print(f'X_train shape : {X_train.shape}')
    print(f'X_test  shape : {X_test.shape}')
    print(f'y_train shape : {y_train.shape}')
    print(f'y_test  shape : {y_test.shape}')

    # Test set class balance
    test_unique, test_counts = np.unique(y_test, return_counts=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Normal', 'Attack'], test_counts, color=['#2196F3', '#F44336'], edgecolor='white')
    for i, (c, v) in enumerate(zip(['Normal', 'Attack'], test_counts)):
        ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
    ax.set_title('Test Set Class Distribution (Processed)', fontsize=13)
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('../results/graphs/test_class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Processed data not found.')
    print('Run  python preprocessing/preprocess.py  first.')

---
## Summary

| Step | Command |
|------|---------|
| Preprocess data | `python preprocessing/preprocess.py` |
| Train model     | `python model/train.py` |
| Make predictions| `python model/predict.py` |

Results and graphs are saved in `results/graphs/` and `results/reports/`.